# In Class Assignment 2026.04.09

In [1]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score

### Load the credit card dataset and perform some initial EDA. In a markdown cell discuss some highlights from your EDA.

In [2]:
df = pd.read_csv("creditcard.csv")

In [3]:
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [6]:
report = sv.analyze(df)
report.show_html("sweet_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Oddly, there is a class imbalance for sex, with 40% of observations having a value of 1, and the remaining 60% having a value of 2.  Additionally, all of the pay_i variables are strongly correlated with each other, and all of the bill_amt_i varaibles are strongly correlated with each other.  However, pay_amt_i variables don't have as strong of a correlation.  As noted in class, the output variable suffers from class imbalance as well, with only 20% of the observations having defaulted on the next month's payment.

### Prepare the data and build default tuned RandomForestClassifier and XGBClassifier models (use the AI-generated defaults for now). Do this with both the entire data set and using 5-fold cross-validation.  Calculate the mean metric score and standard deviation for the folds. In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?


In [4]:
X = df.drop("default payment next month", axis=1)
y = df["default payment next month"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
model_rf = RandomForestClassifier(random_state=42)
model_rf_fitted = model_rf.fit(X_train, y_train)
y_pred_rf = model_rf_fitted.predict(X_test)

In [6]:
model_xgb = XGBClassifier(random_state=42)
model_xgb_fitted = model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb_fitted.predict(X_test)

In [7]:
rf_cv_scores = cross_val_score(model_rf, X_train, y_train, cv=5)
xgb_cv_scores = cross_val_score(model_xgb, X_train, y_train, cv=5)
print("Mean CV Accuracy for Random Forest:", np.mean(rf_cv_scores))
print("Mean CV Accuracy for XGBoost:", np.mean(xgb_cv_scores))
print("Standard Deviation of CV Accuracy for Random Forest:", np.std(rf_cv_scores))
print("Standard Deviation of CV Accuracy for XGBoost:", np.std(xgb_cv_scores))

Mean CV Accuracy for Random Forest: 0.8186458333333334
Mean CV Accuracy for XGBoost: 0.8132291666666667
Standard Deviation of CV Accuracy for Random Forest: 0.004986418185375595
Standard Deviation of CV Accuracy for XGBoost: 0.005786643505137976


In [8]:
print("Recall for Random Forest:", recall_score(y_test, y_pred_rf))
print("Recall for XGBoost:", recall_score(y_test, y_pred_xgb))

Recall for Random Forest: 0.375
Recall for XGBoost: 0.3675925925925926


While both models have similar accuracy (although the random forest model is slightly better), they both have poor recall, meaning that both models struggle to identify default payments correctly.  The low standard deviation indicates that the model performs similarly across a variety of data (or that the data happen to be similar in each split), and the high means indicate that the model is doing an above-average job of determining `default payment next month`

### Look at the classification report for your two models. Does this change your evaluation of the models?



In [9]:
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3720
           1       0.64      0.38      0.47      1080

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest Accuracy: 0.8114583333333333


In [10]:

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      3720
           1       0.60      0.37      0.46      1080

    accuracy                           0.80      4800
   macro avg       0.72      0.65      0.67      4800
weighted avg       0.78      0.80      0.78      4800

XGBoost Accuracy: 0.8025


My evaluation of the models don't change after seeing the classification report, although I notice that the random forest is performing slightly better across every single category.

### Build your models using cross validation again, but this time for both use model features to adjust for the class imbalance. How did this impact model performance?



In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)

In [ ]:
rf_cv_scores = cross_val_score(RandomForestClassifier(random_state=42), X_train, y_train, cv=5)
xgb_cv_scores = cross_val_score(XGBClassifier(random_state=42), X_train, y_train, cv=5)
print("Random Forest Cross-Validation Scores:", rf_cv_scores)
print("XGBoost Cross-Validation Scores:", xgb_cv_scores)

Random Forest Cross-Validation Scores: [0.81666667 0.81328125 0.809375   0.81953125 0.8140625 ]
XGBoost Cross-Validation Scores: [0.80546875 0.81197917 0.80520833 0.81223958 0.81171875]


In [23]:
print("Random Forest Mean CV Score:", np.mean(rf_cv_scores))
print("XGBoost Mean CV Score:", np.mean(xgb_cv_scores))

Random Forest Mean CV Score: 0.8145833333333334
XGBoost Mean CV Score: 0.8093229166666667


In [28]:
rf_recall_scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5, scoring='recall')
xgb_recall_scores = cross_val_score(XGBClassifier(random_state=42), X, y, cv=5, scoring='recall')

In [29]:
print("Random Forest Recall Scores:", rf_recall_scores)
print("XGBoost Recall Scores:", xgb_recall_scores)
print("Random Forest Mean Recall Score:", np.mean(rf_recall_scores))
print("XGBoost Mean Recall Score:", np.mean(xgb_recall_scores))
print("Random Forest Classification Report with recall:")
print(classification_report(y_test, y_pred_rf))
print("XGBoost Classification Report with recall:")
print(classification_report(y_test, y_pred_xgb))

Random Forest Recall Scores: [0.35532516 0.36158192 0.3653484  0.36346516 0.36629002]
XGBoost Recall Scores: [0.35626767 0.37288136 0.37193974 0.36817326 0.37382298]
Random Forest Mean Recall Score: 0.36240213279942346
XGBoost Mean Recall Score: 0.36861699956158334
Random Forest Classification Report with recall:
              precision    recall  f1-score   support

           0       0.78      0.87      0.82      3738
           1       0.22      0.13      0.17      1062

    accuracy                           0.71      4800
   macro avg       0.50      0.50      0.49      4800
weighted avg       0.66      0.71      0.68      4800

XGBoost Classification Report with recall:
              precision    recall  f1-score   support

           0       0.78      0.86      0.82      3738
           1       0.22      0.14      0.17      1062

    accuracy                           0.70      4800
   macro avg       0.50      0.50      0.49      4800
weighted avg       0.65      0.70      0.67

In [31]:
rf_weighted = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_weighted_fitted = rf_weighted.fit(X_train, y_train)
y_pred_rf_weighted = rf_weighted_fitted.predict(X_test)
xgb_weighted = XGBClassifier(random_state=42, scale_pos_weight=1, subsample=0.8)
xgb_weighted_fitted = xgb_weighted.fit(X_train, y_train)
y_pred_xgb_weighted = xgb_weighted_fitted.predict(X_test)
print("Random Forest Classification Report with class weights:")
print(classification_report(y_test, y_pred_rf_weighted))
print("XGBoost Classification Report with class weights:")
print(classification_report(y_test, y_pred_xgb_weighted))

Random Forest Classification Report with class weights:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.65      0.36      0.46      1062

    accuracy                           0.82      4800
   macro avg       0.74      0.65      0.68      4800
weighted avg       0.80      0.82      0.79      4800

XGBoost Classification Report with class weights:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.59      0.36      0.45      1062

    accuracy                           0.80      4800
   macro avg       0.72      0.65      0.67      4800
weighted avg       0.78      0.80      0.79      4800



### Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?



In [13]:
model_xgb_subsample = XGBClassifier(subsample=.8)
model_xgb_subsample_fitted = model_xgb_subsample.fit(X_train, y_train)

In [16]:
cross_val_score(model_xgb_subsample, X_train, y_train, cv=5)
y_pred_subsample = model_xgb_subsample_fitted.predict(X_test)
print("Classification Report for XGBoost model with subsample=.8:")
print(classification_report(y_test, y_pred_subsample))

Classification Report for XGBoost model with subsample=.8:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      3720
           1       0.60      0.37      0.46      1080

    accuracy                           0.80      4800
   macro avg       0.72      0.65      0.67      4800
weighted avg       0.78      0.80      0.78      4800



There doesn't seem to be any noticable difference in performance.

### Do a brief tuning (2 or 3 features) for each model. It is your choice about whether to use the class imbalance adjustments or the subsample feature. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?



In [32]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
}
rf_grid_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring='recall')
rf_grid_search.fit(X_train, y_train)
print("Best Random Forest Parameters:", rf_grid_search.best_params_)
print("Best Random Forest Recall Score:", rf_grid_search.best_score_)

Best Random Forest Parameters: {'max_depth': None, 'n_estimators': 200}
Best Random Forest Recall Score: 0.36872417376844735


In [33]:
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
}
xgb_grid_search = GridSearchCV(XGBClassifier(random_state=42), xgb_param_grid, cv=5, scoring='recall')
xgb_grid_search.fit(X_train, y_train)
print("Best XGBoost Parameters:", xgb_grid_search.best_params_)
print("Best XGBoost Recall Score:", xgb_grid_search.best_score_)

Best XGBoost Parameters: {'max_depth': 6, 'n_estimators': 200}
Best XGBoost Recall Score: 0.37178826300838363


### Which of your models was better out-of-the-box? Be specific.

The random forest model was better, if only slightly, than the xgboost model, in every single category.  The random forest recall and precision was higher for both classes, leading to higher accuracy.  These differences likely aren't statistically significant, so in this regard, it may not be the case that the out-of-box random forest was better.